# QM 640 Capstone — Step 3e: Fetch Text Snippets for Faster Manual Review

This notebook does NOT classify anything. It fetches the actual filing text
for each row still needing review and pastes a readable excerpt directly
into the spreadsheet, so you're reading text in front of you instead of
clicking through to EDGAR and hunting for the relevant paragraph for every
single row. The Y/N and partnership/R&D/M&A decisions are still entirely
yours - this just removes the navigation overhead.

It also filters `screening_recode_sample.csv` down to only the rows that
overlap with what's actually still open for review (it was originally drawn
as 20% of the full ~3,092-row pool, before Steps 3c/3d auto-excluded most of
it - no reason to make your independent re-coder review rows nobody needs).

**Run this after `03d`.**

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [2]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 937, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 937 (delta 54), reused 84 (delta 32), pack-reused 813 (from 1)
Receiving objects: 100% (937/937), 6.91 MiB | 17.33 MiB/s, done.
Resolving deltas: 100% (495/495), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [3]:
!pip install -q pandas requests beautifulsoup4

## Cell 3 — Configuration

In [4]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
REVIEW_FILE = os.path.join(RAW_DIR, "screening_TO_REVIEW.csv")
RECODE_FILE = os.path.join(RAW_DIR, "screening_recode_sample.csv")
HEADERS = {"User-Agent": "QM640 Capstone research Shan_muganathan@yahoo.com"}  # edit to your real email

SNIPPET_LENGTH = 800  # characters of extracted text to show per filing

## Cell 4 — Load files, recover CIK for URL construction

In [5]:
import pandas as pd

master = pd.read_csv(SCREENING_FILE)
review = pd.read_csv(REVIEW_FILE)
recode = pd.read_csv(RECODE_FILE)

# review/recode don't carry "cik" - recover it from the master worksheet
# Ensure 'accession_no' is unique before setting as index to avoid InvalidIndexError
cik_lookup = master.drop_duplicates(subset=["accession_no"]).set_index("accession_no")["cik"]
review["cik"] = review["accession_no"].map(cik_lookup)
recode["cik"] = recode["accession_no"].map(cik_lookup)

print(f"Review file: {len(review)} rows")
print(f"Recode sample (before filtering): {len(recode)} rows")

# Filter the recode sample down to rows that overlap with what's actually
# still open for review - no point re-coding already-excluded rows
still_relevant = recode["accession_no"].isin(review["accession_no"])
recode_filtered = recode[still_relevant].copy()
print(f"Recode sample (filtered to overlap with open review rows): {len(recode_filtered)} rows")

Review file: 814 rows
Recode sample (before filtering): 1883 rows
Recode sample (filtered to overlap with open review rows): 164 rows


## Cell 5 — Fetch and extract a readable text snippet per filing

Direct EDGAR document URL is built from `cik` + the accession/filename
pair already embedded in `accession_no` (format: `accession:filename`).

In [6]:
import requests
import time
import re
from bs4 import BeautifulSoup

def fetch_snippet(cik, accession_no, length=SNIPPET_LENGTH):
    if pd.isna(cik) or not isinstance(accession_no, str) or ":" not in accession_no:
        return "COULD NOT BUILD URL - check filing_url manually"

    accession, filename = accession_no.split(":", 1)
    accession_nodash = accession.replace("-", "")
    url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession_nodash}/{filename}"

    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        if resp.status_code != 200:
            return f"FETCH FAILED ({resp.status_code}) - check filing_url manually"

        soup = BeautifulSoup(resp.text, "html.parser")
        text = soup.get_text(separator=" ", strip=True)
        text = re.sub(r"\s+", " ", text).strip()
        return text[:length] + ("..." if len(text) > length else "")
    except Exception as e:
        return f"FETCH ERROR ({e}) - check filing_url manually"


def add_snippets(df, label):
    snippets = []
    for i, row in df.iterrows():
        snippets.append(fetch_snippet(row["cik"], row["accession_no"]))
        if (i + 1) % 25 == 0:
            print(f"  {label}: {i+1}/{len(df)}")
        time.sleep(0.15)
    df["text_snippet"] = snippets
    return df


print("Fetching snippets for review file ...")
review = add_snippets(review, "review")

print("\nFetching snippets for recode sample ...")
recode_filtered = add_snippets(recode_filtered, "recode")

Fetching snippets for review file ...
  review: 25/814
  review: 50/814
  review: 75/814
  review: 100/814
  review: 125/814
  review: 150/814
  review: 175/814
  review: 200/814
  review: 225/814
  review: 250/814
  review: 275/814
  review: 300/814
  review: 325/814
  review: 350/814
  review: 375/814
  review: 400/814
  review: 425/814
  review: 450/814
  review: 475/814
  review: 500/814
  review: 525/814
  review: 550/814
  review: 575/814
  review: 600/814
  review: 625/814
  review: 650/814
  review: 675/814
  review: 700/814
  review: 725/814
  review: 750/814
  review: 775/814
  review: 800/814

Fetching snippets for recode sample ...
  recode: 225/164
  recode: 350/164
  recode: 1200/164
  recode: 1650/164


## Cell 6 — Save both files with snippets attached

In [7]:
# Reorder so the snippet sits right next to the columns you're filling in
review_cols = ["accession_no", "company_name", "file_date", "text_snippet",
               "announcement_type", "is_genuine_ai_event", "item_codes",
               "item_in_scope", "filing_url"]
review = review[review_cols]
import csv
review.to_csv(REVIEW_FILE, index=False, quoting=csv.QUOTE_ALL)  # robust against Excel/Sheets CSV round-trip corruption
print(f"Saved -> {REVIEW_FILE}")

recode_cols = ["accession_no", "company_name", "file_date", "text_snippet",
               "recoder_announcement_type", "recoder_is_genuine_ai_event"]
recode_filtered = recode_filtered[recode_cols]
recode_filtered.to_csv(RECODE_FILE, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved (filtered + snippeted) -> {RECODE_FILE}")

Saved -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_TO_REVIEW.csv
Saved (filtered + snippeted) -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_recode_sample.csv


## Commit and push results back to GitHub

In [8]:
!git -C {BASE_DIR} add "data/raw/screening_TO_REVIEW.csv"
!git -C {BASE_DIR} add "data/raw/screening_recode_sample.csv"
!git -C {BASE_DIR} commit -m "Step 3e: add text snippets for faster manual review, filter recode sample"
!git -C {BASE_DIR} push

[main cd0e5a4] Step 3e: add text snippets for faster manual review, filter recode sample
 2 files changed, 980 insertions(+), 2699 deletions(-)
 rewrite data/raw/screening_TO_REVIEW.csv (89%)
 rewrite data/raw/screening_recode_sample.csv (92%)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (6/6), 297.88 KiB | 5.73 MiB/s, done.
Total 6 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   7a0c5c7..cd0e5a4  main -> main


## What to do now

Open `screening_TO_REVIEW.csv` — you'll see a `text_snippet` column right
there in the sheet. For most rows, that's enough to decide
`is_genuine_ai_event` and `announcement_type` without clicking anything.
Only click through to `filing_url` when the snippet is ambiguous, cut off
mid-sentence, or shows a fetch error.

Same workflow for your independent reviewer on the (now much smaller)
`screening_recode_sample.csv`.